In [1]:
from groq import Groq
from dotenv import load_dotenv
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [2]:
stream = response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages = [{"role": "user", "content": "tell me a bad time story about a unicorn"}],
    stream= True
)

response = None
events = []
for event in stream:
    events.append(event)
    if event.choices[0].delta.content:
        print(event.choices[0].delta.content, end='')

Once upon a time, in a land far, far away, there was a unicorn named Gary. Gary was a bit of an oddball in the unicorn world. While most unicorns were known for their beautiful, shimmering coats and their ability to poop rainbows, Gary's coat was more of a dull gray, and his poop was just...well, regular poop.

One day, Gary decided to enter the annual Unicorn Talent Show, hoping to prove to the other unicorns that he was more than just a weird-looking outcast. He spent weeks practicing his magic, but no matter how hard he tried, he just couldn't seem to get it right.

The day of the talent show arrived, and Gary took the stage in front of a crowd of snickering unicorns. He raised his horn, closed his eyes, and attempted to conjure up a beautiful burst of sparkles. But instead of sparkles, a giant fart escaped from his rear end, causing the audience to erupt in laughter.

The judges, a panel of wise old unicorns, were not amused. "Gary, Gary, Gary," they said, shaking their heads. "You

In [3]:
from pydantic import BaseModel, Field

In [4]:
class ArticleSection(BaseModel):
    header: str
    text: str = Field(description="text of the section in markdown")

class ArticleResponse(BaseModel):
    title: str
    subtitle: str
    sections: list[ArticleSection]

In [ ]:
import json

schema = ArticleResponse.model_json_schema()

stream = response = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages = [{"role": "system", "content": f"Extract the event information. Return JSON only matching this schema: {schema}"},
                {"role": "user", "content": "tell me a bad time story about a unicorn"}
                ],
    stream= True,
    response_format={"type": "json_object"}
)

response = None
events = []
raw = ""
for event in stream:
    events.append(event)
    chunk = event.choices[0].delta.content
    if event.choices[0].delta.content:
        print(event.choices[0].delta.content, end='')
        raw += chunk

event = ArticleResponse(**json.loads(raw))
print(event)

{
  "title": "The Dark Unicorn",
   "subtitle": "A Tale of Woe and Sorrow",
   "sections": [
       {
           "header": "The Fall of Sparkles",
           "text": "In a land far, far away, there was a unicorn named Sparkles. Sparkles was once a beautiful and majestic creature, with a coat as white as snow and a horn that shone like the brightest diamond. But one fateful day, Sparkles stumbled upon a dark and mysterious forest, and everything changed. The forest was home to a wicked sorcerer who cast a spell on Sparkles, turning its horn a deep, foreboding black."
       },
       {
           "header": "The Descent into Darkness",
           "text": "As time passed, Sparkles became increasingly consumed by the darkness within. It began to use its powers for evil, casting spells on the other creatures of the forest and bringing terror to the land. The once-peaceful forest was now a place of fear and dread, and Sparkles was the source of it all. The other unicorns, who had once looked

In [8]:
story = event

print('# ' + story.title)
print(story.subtitle)
print()

for section in story.sections:
    print('## ' + section.header)
    print()
    print(section.text)
    print()
    print()

# The Dark Unicorn
A Tale of Woe and Sorrow

## The Fall of Sparkles

In a land far, far away, there was a unicorn named Sparkles. Sparkles was once a beautiful and majestic creature, with a coat as white as snow and a horn that shone like the brightest diamond. But one fateful day, Sparkles stumbled upon a dark and mysterious forest, and everything changed. The forest was home to a wicked sorcerer who cast a spell on Sparkles, turning its horn a deep, foreboding black.


## The Descent into Darkness

As time passed, Sparkles became increasingly consumed by the darkness within. It began to use its powers for evil, casting spells on the other creatures of the forest and bringing terror to the land. The once-peaceful forest was now a place of fear and dread, and Sparkles was the source of it all. The other unicorns, who had once looked up to Sparkles as a symbol of hope and magic, now shunned it and feared its power.


## The Final Confrontation

One day, a brave knight named Sir Valoric

In [9]:
!uv add jaxn

Resolved 122 packages in 860ms                                       
⠙ Preparing packages... (0/1)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/13.08 KiB           
⠙ Preparing packages... (0/1)---------- 13.08 KiB/13.08 KiB         
Prepared 1 package in 122ms                                                       
░░░░░░░░░░░░░░░░░░░░ [0/1] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 35ms                                 
 + jaxn==0.0.5


In [10]:
from jaxn import JSONParserHandler, StreamingJSONParser

In [11]:
raw_json = """
{"title":"The Misadventures of Sparkle","subtitle":"A Tale of Trouble","sections":[{"header":"Once Upon a Time","text":"In a magical land far away, there lived a young unicorn named Sparkle."},{"header":"The Problem","text":"Sparkle had a mischievous streak that often led to chaos."}]}
""".strip()


In [12]:
from typing import Any, Dict

class ArticleResponseHandler(JSONParserHandler):

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '':
            if field_name == 'title':
                print(f'# {value}')
            if field_name == 'subtitle':
                print(value)
        if path == '/sections' and field_name == 'header':
            print(f'\n\n## {value}\n')

    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '/sections' and field_name == 'text':
            print(chunk, end='', flush=True)

In [13]:
handler = ArticleResponseHandler()
parser = StreamingJSONParser(handler=handler)


In [14]:
for c in raw_json:
    parser.parse_incremental(c)

# The Misadventures of Sparkle
A Tale of Trouble


## Once Upon a Time

In a magical land far away, there lived a young unicorn named Sparkle.

## The Problem

Sparkle had a mischievous streak that often led to chaos.

In [16]:
def llm_structured_stream(user_prompt, output_type, parser_handler=JSONParserHandler(), instructions=None, model="llama-3.3-70b-versatile"):
    schema = output_type.model_json_schema()
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions + f"\n\nYour response must be a JSON object with the following structure:\n{schema}\n\nReturn only the JSON object with actual values, not the schema itself."
        })
    
    messages.append({
        "role": "user",
        "content": user_prompt
    })
    
    parser = StreamingJSONParser(handler=parser_handler)
    
    stream = groq_client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
        response_format={"type": "json_object"}
    )

    raw = ""
    for event in stream:
        chunk = event.choices[0].delta.content
        if chunk:
            parser.parse_incremental(chunk)
            raw += chunk

    result = output_type(**json.loads(raw))
    return result

In [17]:
instructions = "your task is to tell the user bad time stories"
user_prompt =  "unicorn"

result = llm_structured_stream(
    instructions=instructions,
    user_prompt=user_prompt,
    output_type=ArticleResponse,
    parser_handler=ArticleResponseHandler(),
)

# The Curse of the Unicorn
A Dark and Troubling Tale


## The Encounter

It was a dark and stormy night when I first encountered the unicorn. Its horn glowed with an otherworldly light, illuminating the path ahead. But as I drew closer, I realized that its eyes were black as coal, and its voice was like a rusty gate.

## The Curse

The unicorn spoke to me in a voice that sent shivers down my spine. It told me that I was cursed, that I would never know happiness again. And with that, it vanished into thin air, leaving me to ponder the truth of its words.

## The Descent into Madness

From that day on, I was plagued by dark visions and terrible nightmares. I became convinced that the unicorn was watching me, waiting for me to succumb to the curse. And as the days turned into weeks, I felt my grip on reality begin to slip.

In [18]:
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results



instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""


prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

Indexed 385 chunks from 95 documents


In [19]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [20]:
class RAGResponseHandler(JSONParserHandler):
    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '' and field_name == 'answer':
            print(chunk, end='', flush=True)

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '' and field_name == 'answer_type':
            print('\nanswer type:', value)

    def on_array_item_end(self, path: str, field_name: str, item: Dict[str, Any] = None) -> None:
        if field_name == 'followup_questions':
            print('follow up question:', item)

In [21]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured_stream(
        instructions=instructions,
        user_prompt=prompt,
        output_type=RAGResponse,
        parser_handler=RAGResponseHandler()
    )

In [22]:
response = rag('llm as a judge')

To use an LLM as a judge, you can create and evaluate it using the Evidently library. There are two ways to use an LLM as a judge: **reference-based** and **open-ended**. The reference-based approach compares new responses against a reference, while the open-ended approach evaluates responses based on custom criteria. You can create and tune the LLM evaluator using a toy Q&A dataset and then apply it in different contexts, such as regression testing or prompt comparison.
answer type: explanation
follow up question: What are the requirements for creating and evaluating an LLM judge?
follow up question: How can I tune the LLM evaluator for better results?
follow up question: What are the differences between reference-based and open-ended approaches to using an LLM as a judge?
